# Debug S3 Temporary Prefix

Notebook to inspect and optionally clean the temporary S3 input folder used by train preprocessing.


In [ ]:
%load_ext autoreload
%load_ext dotenv
%dotenv
%autoreload 2

from pathlib import Path
from datetime import datetime
import os
import sys

PROJECT_ROOT = Path(os.getenv('PROJECT_ROOT_PATH'))
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from cvm_model.io import State
import cvm_model.utils as utils
from cvm_model.parameters import input_suffix


In [ ]:
state = State.from_env()
s3 = state.credentials.cvm_s3
print('State loaded')


## Inspect Full Temporary Folder

This shows what is currently stored under `state.settings.s3_temporary_prefix` — the full temporary project folder, not only the preprocess input subfolder.


In [ ]:
temporary_root = state.settings.s3_temporary_prefix
temporary_bucket = temporary_root.split('//')[1].split('/')[0]
temporary_prefix = '/'.join(temporary_root.split('//')[1].split('/')[1:])
temporary_path = f'{temporary_bucket}/{temporary_prefix.strip("/")}'

print('temporary_root:', temporary_root)
print('temporary_bucket:', temporary_bucket)
print('temporary_prefix:', temporary_prefix)
print('temporary_path:', temporary_path)


In [ ]:
# Top-level objects/folders under temporary root.
fs = s3.s3fs

if fs.exists(temporary_path):
    top_level = fs.ls(temporary_path, detail=False)
else:
    top_level = []

print('top-level count:', len(top_level))
for obj in top_level[:100]:
    print(obj)

if len(top_level) > 100:
    print(f'... and {len(top_level) - 100} more')


In [ ]:
# Recursive listing can be large; use MAX_OBJECTS to avoid flooding the notebook.
MAX_OBJECTS = 300

if fs.exists(temporary_path):
    all_objects = fs.find(temporary_path)
else:
    all_objects = []

print('recursive objects count:', len(all_objects))
for obj in all_objects[:MAX_OBJECTS]:
    print(obj)

if len(all_objects) > MAX_OBJECTS:
    print(f'... and {len(all_objects) - MAX_OBJECTS} more')


## Optional Cleanup For Full Temporary Folder

Be careful: this removes the whole temporary prefix, not just the current preprocess input folder.


In [ ]:
CLEAN_FULL_TEMPORARY = False

if CLEAN_FULL_TEMPORARY:
    utils.remove_s3_prefix(s3, temporary_bucket, temporary_prefix)
    objects_after = utils.list_s3_objects(s3, temporary_bucket, temporary_prefix)
    print('objects after full temporary cleanup:', len(objects_after))
    assert len(objects_after) == 0, 'Temporary prefix was not cleaned'
else:
    print('Full temporary cleanup skipped. Set CLEAN_FULL_TEMPORARY = True to remove it.')


## Choose Event Timestamp

Use the same `event_timestamp` that is passed to `preprocess_train`.


In [ ]:
event_timestamp = datetime(2026, 2, 1)


## Resolve S3 Prefix

This follows the same logic as `preprocess.py`: `state.settings.preprocess_prefix(event_timestamp) / input_suffix`.


In [ ]:
input_prefix = state.settings.preprocess_prefix(event_timestamp) / input_suffix
input_bucket = input_prefix.split('//')[1].split('/')[0]

print('input_prefix:', input_prefix)
print('input_bucket:', input_bucket)
print('train prefix:', f'{input_prefix}/train')
print('holdout prefix:', f'{input_prefix}/holdout')


## List Current Objects


In [ ]:
objects = utils.list_s3_objects(s3, input_bucket, input_prefix)

print('objects count:', len(objects))
for obj in objects[:50]:
    print(obj)

if len(objects) > 50:
    print(f'... and {len(objects) - 50} more')


## Optional Cleanup

Set `DO_CLEAN = True` only after checking the listed objects above.


In [ ]:
DO_CLEAN = False

if DO_CLEAN:
    utils.remove_s3_prefix(s3, input_bucket, input_prefix)
    objects_after = utils.list_s3_objects(s3, input_bucket, input_prefix)
    print('objects after cleanup:', len(objects_after))
    assert len(objects_after) == 0, 'S3 prefix was not cleaned'
else:
    print('Cleanup skipped. Set DO_CLEAN = True to remove this prefix.')


## Check Again


In [ ]:
objects = utils.list_s3_objects(s3, input_bucket, input_prefix)
print('objects count:', len(objects))
for obj in objects[:50]:
    print(obj)
